# Testing => Unit Testing With `unittest`

A **unit test** checks one small piece of code automatically. `unittest` is Python's built-in testing framework.

| Tool | Purpose |
|---|---|
| `unittest.TestCase` | Base class. Each method named `test_*` is one test |
| `assertEqual(a, b)` | `a == b` |
| `assertTrue(x)` / `assertFalse(x)` | Truth checks |
| `assertIs(a, b)` / `assertIsNone(x)` | Identity checks |
| `assertIn(a, b)` | `a in b` |
| `assertAlmostEqual(a, b)` | Floats that are nearly equal |
| `assertRaises(Error)` | The block must raise `Error` |
| `setUp()` / `tearDown()` | Run before / after **each** test |
| `setUpClass()` / `tearDownClass()` | Run once per class |
| `subTest()` | Run several cases inside one test |
| `@unittest.skip(...)` | Skip a test |
| `python -m unittest` | Discover and run tests |

---

## Structure

```python
import unittest

def add(a, b):
    return a + b

class TestAdd(unittest.TestCase):
    def test_positive(self):
        self.assertEqual(add(2, 3), 5)

    def test_negative(self):
        self.assertEqual(add(-1, -1), -2)

if __name__ == "__main__":
    unittest.main()
```

* Test methods must start with `test`.
* Each test is **independent**. Do not rely on test order.
* A test **passes** if no exception is raised. It **fails** on `AssertionError`. Any other exception is an **error**.

---

## Testing Exceptions

```python
with self.assertRaises(ZeroDivisionError):
    divide(1, 0)
```

---

## Fixtures: `setUp` and `tearDown`

```python
class TestStack(unittest.TestCase):
    def setUp(self):
        self.stack = []          # fresh for every test

    def tearDown(self):
        ...                      # clean up
```

---

## Several Cases: `subTest`

```python
for a, b, expected in cases:
    with self.subTest(a=a, b=b):
        self.assertEqual(add(a, b), expected)
```

A failing case is reported separately and does not stop the others.

---

## Running Tests

| Where | How |
|---|---|
| Terminal | `python -m unittest` (discovers `test*.py` files) |
| A file | `python -m unittest test_math.py` |
| A single test | `python -m unittest test_math.TestAdd.test_positive` |
| Notebook | `unittest.main(argv=[""], exit=False)` |

---

## `unittest` vs `pytest`

| | `unittest` | `pytest` (third-party) |
|---|---|---|
| Installation | Built in | `pip install pytest` |
| Test style | Classes and `self.assert...` | Plain functions and `assert` |
| Shared setup | `setUp` / `tearDown` | Fixtures |
| Many cases | `subTest` | `@pytest.mark.parametrize` |
| Discovery | `python -m unittest` | `pytest` |

`pytest` also runs `unittest` tests, so learning `unittest` first is not wasted.

## Source

https://docs.python.org/3/library/unittest.html

https://docs.pytest.org/

In [ ]:
import io
import unittest

def add(a, b):
    return a + b

def divide(a, b):
    return a / b

class TestMath(unittest.TestCase):
    def setUp(self):
        self.numbers = [1, 2, 3]                    # fresh for every test

    def test_add(self):
        self.assertEqual(add(2, 3), 5)

    def test_float(self):
        self.assertAlmostEqual(0.1 + 0.2, 0.3)

    def test_membership(self):
        self.assertIn(2, self.numbers)
        self.assertIsNone(None)
        self.assertTrue(add(1, 1) == 2)

    def test_error(self):
        with self.assertRaises(ZeroDivisionError):
            divide(1, 0)

    def test_many_cases(self):
        cases = [(1, 1, 2), (0, 0, 0), (-1, 1, 0)]
        for a, b, expected in cases:
            with self.subTest(a=a, b=b):
                self.assertEqual(add(a, b), expected)

    @unittest.skip("not written yet")
    def test_skipped(self):
        self.fail("never runs")

class TestFailing(unittest.TestCase):
    def test_wrong(self):
        self.assertEqual(add(2, 2), 5)              # fails on purpose

def run(test_case):
    """Run one TestCase class and return the result (output is captured)."""
    suite = unittest.TestLoader().loadTestsFromTestCase(test_case)
    return unittest.TextTestRunner(stream=io.StringIO(), verbosity=0).run(suite)

good = run(TestMath)
print(good.testsRun, good.wasSuccessful(), len(good.failures), len(good.skipped))

bad = run(TestFailing)
print(bad.testsRun, bad.wasSuccessful(), len(bad.failures))
print(bad.failures[0][1].strip().splitlines()[-1])   # the assertion message

# In a notebook you can also run every test class in the current namespace:
#     unittest.main(argv=[""], exit=False)